# District rainfall folder: usefulness checks
Read-only check of user-supplied files. Source page references are retained but have not been reconciled against the original publication. Rounding bounds assume values rounded independently to nearest 0.1 mm; exceeding the bound flags reconciliation, not an automatic repair. No blanks are converted to zero.

In [1]:
from pathlib import Path
import csv, collections, math, json
states_dir = Path.home() / "Downloads" / "states"
files = sorted(states_dir.glob("*.csv"))
rows = [r for f in files for r in csv.DictReader(f.open(encoding="utf-8-sig"))]
months = [m + "_mm" for m in ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"]]
keys = collections.Counter((r["series_id"],r["year"]) for r in rows)
ahmedabad = [r for r in rows if r["state_source"] == "Gujarat" and r["district_source"] == "Ahmedabad"]
checks = [("annual_mm",months,.65),("jf_mm",months[:2],.15),("mam_mm",months[2:5],.20),("jjas_mm",months[5:9],.25),("ond_mm",months[9:],.20)]
anomalies = []
for field, components, tolerance in checks:
    for r in rows:
        if all(r[k] for k in [field] + components):
            delta = float(r[field]) - sum(float(r[k]) for k in components)
            if abs(delta) > tolerance + 1e-8:
                anomalies.append({"series_id":r["series_id"], "district":r["district_source"], "year":int(r["year"]), "field":field, "difference_mm":round(delta,4)})
result = {"files":len(files), "rows":len(rows), "series":len(set(r["series_id"] for r in rows)),
          "common_schema":len(set(tuple(r) for r in rows)) == 1,
          "year_range":[min(int(r["year"]) for r in rows),max(int(r["year"]) for r in rows)],
          "duplicate_series_year_keys":sum(v>1 for v in keys.values()),
          "rows_missing_months":sum(any(not r[m] for m in months) for r in rows),
          "missing_month_cells":sum(not r[m] for r in rows for m in months),
          "invalid_numeric_values":sum(not math.isfinite(float(v)) or float(v)<0 for r in rows for k,v in r.items() if k.endswith("_mm") and v),
          "flag_mismatches":sum(("missing_months" in r["quality_flags"]) != any(not r[m] for m in months) for r in rows),
          "rows_with_duplicate_source_pages":sum(bool(r["duplicate_source_pages"]) for r in rows),
          "ahmedabad":{"rows":len(ahmedabad),"years":[min(int(r["year"]) for r in ahmedabad),max(int(r["year"]) for r in ahmedabad)],
                       "missing_months":sum(not r[m] for r in ahmedabad for m in months),
                       "source_pages":sorted(set(r["source_page"] for r in ahmedabad))},
          "aggregate_discrepancies_beyond_rounding_bounds":anomalies}
print(json.dumps(result,indent=2))


{
  "files": 32,
  "rows": 60568,
  "series": 640,
  "common_schema": true,
  "year_range": [
    1901,
    2010
  ],
  "duplicate_series_year_keys": 0,
  "rows_missing_months": 5018,
  "missing_month_cells": 15817,
  "invalid_numeric_values": 0,
  "flag_mismatches": 0,
  "rows_with_duplicate_source_pages": 294,
  "ahmedabad": {
    "rows": 110,
    "years": [
      1901,
      2010
    ],
    "missing_months": 0,
    "source_pages": [
      "611",
      "612",
      "613"
    ]
  },
  "aggregate_discrepancies_beyond_rounding_bounds": [
    {
      "series_id": "IMD110-P1403",
      "district": "Kendrapada",
      "year": 1920,
      "field": "annual_mm",
      "difference_mm": 12.5
    },
    {
      "series_id": "IMD110-P1403",
      "district": "Kendrapada",
      "year": 1920,
      "field": "ond_mm",
      "difference_mm": 12.5
    }
  ]
}
